# eBay Market Price Research

Fetch sold and active eBay listings for a MTG card and analyze the price distribution.


In [11]:
import httpx
import pandas as pd
import matplotlib.pyplot as plt


BASE_URL = "https://automana.duckdns.org/"

# ── Card inputs — edit these ──────────────────────────────────────────────
APP_CODE     = "automana_au"      # your eBay app code
CARD_NAME    = "Sheoldred, the Apocalypse"
SET_CODE     = "DMR"
IS_FOIL      = False
FRAME        = None               # None | 'showcase' | 'extended_art' | 'borderless'
CONDITION_ID = None               # None | 3000 (NM) | 4000 (LP) | 5000 (MP)
DAYS_BACK    = 30
LIMIT        = 50
THRESHOLD    = 0.6


In [15]:
params = {
    "card_name": CARD_NAME,
    "app_code": APP_CODE,
    "set_code": SET_CODE,
    "is_foil": IS_FOIL,
    "days_back": DAYS_BACK,
    "limit": LIMIT,
    "match_threshold": THRESHOLD,
}
if FRAME:        params["frame"] = FRAME
if CONDITION_ID: params["condition_id"] = CONDITION_ID

resp = httpx.get(f"{BASE_URL}/api/v1/integrations/ebay/market-price/", params=params, timeout=30)
print(resp.text[:1000])
resp.raise_for_status()
data = resp.json()

print(f"Card:  {data['card_name']}  |  Set: {data['set_code']}")
print(f"Query: {data['query']}")
print(f"Sold results: {data['sold_aggregates']['count']}   Active results: {data['active_aggregates']['count']}")
print(f"Suggested price: {data['suggested_price']}")


<!doctype html>
<html lang="en">
  <head>
    <script type="module">import { injectIntoGlobalHook } from "/@react-refresh";
injectIntoGlobalHook(window);
window.$RefreshReg$ = () => {};
window.$RefreshSig$ = () => (type) => type;</script>

    <script type="module" src="/@vite/client"></script>

    <meta charset="UTF-8" />
    <meta name="viewport" content="width=device-width, initial-scale=1.0" />
    <title>automana</title>
  </head>
  <body>
    <div id="root"></div>
    <script type="module" src="/src/main.tsx"></script>
  </body>
</html>



JSONDecodeError: Expecting value: line 1 column 1 (char 0)

In [ ]:
sold_df   = pd.DataFrame(data["sold"])   if data["sold"]   else pd.DataFrame()
active_df = pd.DataFrame(data["active"]) if data["active"] else pd.DataFrame()

if not sold_df.empty:
    sold_df = sold_df.sort_values("relevance_score", ascending=False)
    print("=== Sold listings (top 10) ===")
    display(sold_df[["title", "price", "currency", "condition", "sold_date", "relevance_score"]].head(10))

if not active_df.empty:
    active_df = active_df.sort_values("relevance_score", ascending=False)
    print("=== Active listings (top 10) ===")
    display(active_df[["title", "price", "currency", "condition", "relevance_score"]].head(10))


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle(f"{data['card_name']} ({data.get('set_code', '')}) — Price Distribution", fontsize=14)

def _plot_hist(ax, df, col, label, color, agg):
    if df.empty or col not in df.columns:
        ax.text(0.5, 0.5, "No data", ha="center", va="center", transform=ax.transAxes)
        ax.set_title(f"{label} (n=0)")
        return
    ax.hist(df[col], bins=15, color=color, alpha=0.75, edgecolor="white")
    median = agg.get("median")
    p25    = agg.get("p25")
    p75    = agg.get("p75")
    if median is not None:
        ax.axvline(median, color="navy" if color == "steelblue" else "darkred",
                   linestyle="--", linewidth=1.8, label=f"Median ${median:.2f}")
    if p25 is not None and p75 is not None:
        ax.axvspan(p25, p75, alpha=0.12, color=color, label=f"IQR ${p25:.2f}–${p75:.2f}")
    ax.set_title(f"{label} (n={agg.get('count', 0)})")
    ax.set_xlabel("Price (AUD)")
    ax.set_ylabel("Count")
    ax.legend(fontsize=8)

_plot_hist(axes[0], sold_df,   "price", "Sold (completed)",  "steelblue", data["sold_aggregates"])
_plot_hist(axes[1], active_df, "price", "Active (listed now)", "coral",   data["active_aggregates"])

plt.tight_layout()
plt.show()


In [ ]:
s = data["sold_aggregates"]
a = data["active_aggregates"]
sp = data["suggested_price"]

print("╔══════════════════════════════════════╗")
print("║       PRICE SUGGESTION SUMMARY       ║")
print("╠══════════════════════════════════════╣")
if sp:
    print(f"║  Suggested price (sold median): ${sp:>6.2f} ║")
else:
    print("║  Suggested price: N/A (<3 sold results)║")
print(f"║  Sold  — n={s['count']:<3}  median=${s.get('median') or 0:>6.2f}        ║")
print(f"║         IQR  ${s.get('p25') or 0:.2f} – ${s.get('p75') or 0:.2f}              ║")
print(f"║  Active— n={a['count']:<3}  floor=${a.get('min') or 0:>6.2f}         ║")
print("╚══════════════════════════════════════╝")


In [ ]:
# Threshold tuning — re-run with a lower threshold to see impact
LOW_THRESHOLD = 0.4
params_loose = {**params, "match_threshold": LOW_THRESHOLD}
resp_loose = httpx.get(f"{BASE_URL}/api/v1/integrations/ebay/market-price/", params=params_loose, timeout=30)
resp_loose.raise_for_status()
data_loose = resp_loose.json()

print(f"threshold={THRESHOLD}:      sold={data['sold_aggregates']['count']}  active={data['active_aggregates']['count']}")
print(f"threshold={LOW_THRESHOLD}: sold={data_loose['sold_aggregates']['count']}  active={data_loose['active_aggregates']['count']}")
print(f"Extra sold results at lower threshold: {data_loose['sold_aggregates']['count'] - data['sold_aggregates']['count']}")
